# RiskModels — stocks & portfolios (SDK golden path)

**[Get API key](https://riskmodels.app/get-key)** · **[ERM3 methodology](https://riskmodels.net/docs/methodology/erm3-l3)** · **[Open in Colab](https://colab.research.google.com/github/BlueWaterCorp/RiskModels_API/blob/main/sdk/notebooks/surface_stocks_sdk.ipynb)**

Thin onboarding notebook: one ticker **metrics** row, then a **two-name portfolio** `analyze_portfolio()` (alias **`analyze()`**). For REST wire format + AOM, use **`riskmodels_quickstart.ipynb`** in this folder.

**Hedge ratio convention:** dollars of ETF per **$1** of stock (`dollar_ratio`). ER values are **variance shares** in \([0,1]\) at L3.


### Colab only (skip locally)

Run once to install the SDK; local users should `pip install riskmodels-py` in their venv.


In [ ]:
import sys

try:
    import google.colab  # noqa: F401
    _COLAB = True
except ImportError:
    _COLAB = False

if _COLAB:
    import subprocess

    _deps = ["python-dotenv"]
    _pypi = "riskmodels-py>=0.3.4"
    _git = (
        "riskmodels-py @ git+https://github.com/BlueWaterCorp/RiskModels_API.git"
        "@main#subdirectory=sdk"
    )
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", _pypi, *_deps],
            stdout=subprocess.DEVNULL,
        )
        print("Colab: installed riskmodels-py from PyPI (+ python-dotenv).")
    except subprocess.CalledProcessError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", _git, *_deps],
            stdout=subprocess.DEVNULL,
        )
        print(
            "Colab: installed riskmodels-py from GitHub main "
            "(PyPI may not list this version yet)."
        )
else:
    print("Local: use your environment (pip install riskmodels-py python-dotenv).")


## 1. Connect — `RiskModelsClient.from_env()`

Loads **`RISKMODELS_API_KEY`** from shell env, `.env` / `.env.local` (when `python-dotenv` is installed), or Colab Secrets.


In [ ]:
import os

from IPython.display import display

from riskmodels import RiskModelsClient
from riskmodels.client import DEFAULT_BASE_URL
from riskmodels.notebook import load_notebook_dotenv

load_notebook_dotenv()
print("RISKMODELS_BASE_URL =", os.environ.get("RISKMODELS_BASE_URL", DEFAULT_BASE_URL))
client = RiskModelsClient.from_env()


## 2. Single stock — `get_metrics` → DataFrame

Semantic column names (`l3_market_hr`, `l3_residual_er`, …). Raw wire keys live under JSON `metrics` only.


In [ ]:
TICKER = "NVDA"

df = client.get_metrics(TICKER, as_dataframe=True)
display(df)
legend = df.attrs.get("legend") or ""
print(legend[:1200] if legend else "(no legend attr)")


## 3. Portfolio — `analyze_portfolio` / `analyze()`

**Portfolio hedge ratios** are **holdings-weighted means** of per-ticker scalars (not a full optimizer).


In [ ]:
positions = {"NVDA": 0.6, "AAPL": 0.4}
pa = client.analyze_portfolio(positions)

print("Portfolio L3 HR (wmean):", pa.portfolio_hedge_ratios)
print("Portfolio L3 ER (weighted mean):", pa.portfolio_l3_er_weighted_mean)
print()
print(pa.to_llm_context()[:2500])


## Next steps

- **Deeper tutorial:** [`riskmodels_quickstart.ipynb`](./riskmodels_quickstart.ipynb) (REST + SDK + AOM).
- **Contract:** [OpenAPI](https://github.com/BlueWaterCorp/RiskModels_API/blob/main/OPENAPI_SPEC.yaml) / `riskmodels.app/docs`.
- **Funds / 13F / benchmarks (HTTP):** [`surface_funds_http.ipynb`](./surface_funds_http.ipynb) — no funds SDK in `riskmodels-py` yet.
